In [61]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [62]:
df = pd.read_csv("../data/cleaned_data.csv")
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [63]:
df.shape
df.columns
df.info()
df["Churn"].value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [64]:
# Target encode
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

# One-hot encoding
df = pd.get_dummies(df, drop_first=True)

# kontrol
df.dtypes
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_Yes,...,DeviceProtection_Yes,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,False,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,True,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,False,...,True,True,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


In [65]:
x = df.drop("Churn", axis=1)
y = df["Churn"]

In [66]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [67]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [68]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, Y_train)

y_pred_knn = knn.predict(X_test_scaled)

In [69]:
from sklearn.metrics import accuracy_score, classification_report

print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn))

KNN Accuracy: 0.7714691270404542
              precision    recall  f1-score   support

           0       0.83      0.86      0.85      1036
           1       0.58      0.52      0.55       373

    accuracy                           0.77      1409
   macro avg       0.70      0.69      0.70      1409
weighted avg       0.76      0.77      0.77      1409



In [70]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000, class_weight="balanced")
log_model.fit(X_train_scaled, Y_train)

y_pred_log = log_model.predict(X_test_scaled)

In [71]:
print("Logistic Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

Logistic Accuracy: 0.7487579843860894
              precision    recall  f1-score   support

           0       0.92      0.72      0.81      1036
           1       0.52      0.82      0.63       373

    accuracy                           0.75      1409
   macro avg       0.72      0.77      0.72      1409
weighted avg       0.81      0.75      0.76      1409



In [72]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, Y_train)

# Logistic (SMOTE ile)
log_smote = LogisticRegression(max_iter=1000)
log_smote.fit(X_train_res, y_train_res)

y_pred_log_smote = log_smote.predict(X_test_scaled)

print("Logistic + SMOTE")
print(classification_report(y_test, y_pred_log_smote))

Logistic + SMOTE
              precision    recall  f1-score   support

           0       0.92      0.73      0.81      1036
           1       0.52      0.83      0.64       373

    accuracy                           0.76      1409
   macro avg       0.72      0.78      0.73      1409
weighted avg       0.82      0.76      0.77      1409



In [73]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11]
}

grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
grid_knn.fit(X_train_scaled, Y_train)

print(grid_knn.best_params_)

{'n_neighbors': 11}


In [74]:
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100]
}

grid_log = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=5
)

grid_log.fit(X_train_scaled, Y_train)

print(grid_log.best_params_)

{'C': 10}


In [75]:
from sklearn.ensemble import RandomForestClassifier

rf_bal = RandomForestClassifier(class_weight="balanced", random_state=42)
rf_bal.fit(X_train, Y_train)  # RF için scaling gerekmiyor

y_pred_rf_bal = rf_bal.predict(X_test)

print("RandomForest (balanced)")
print(classification_report(y_test, y_pred_rf_bal))

RandomForest (balanced)
              precision    recall  f1-score   support

           0       0.82      0.91      0.87      1036
           1       0.65      0.45      0.53       373

    accuracy                           0.79      1409
   macro avg       0.74      0.68      0.70      1409
weighted avg       0.78      0.79      0.78      1409



In [76]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(log_model, X_train_scaled, Y_train, cv=5)

print(scores)
print("Mean:", scores.mean())

[0.7515528  0.74179237 0.72937001 0.7284827  0.75044405]
Mean: 0.7403283840372198


In [77]:
best_k = grid_knn.best_params_["n_neighbors"]

knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_scaled, Y_train)

y_pred_knn_best = knn_best.predict(X_test_scaled)

print("KNN (best params)")
print(classification_report(y_test, y_pred_knn_best))

KNN (best params)
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1036
           1       0.63      0.54      0.58       373

    accuracy                           0.79      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.79      0.79      1409



In [78]:
best_C = grid_log.best_params_["C"]

log_best = LogisticRegression(max_iter=1000, C=best_C, class_weight="balanced")
log_best.fit(X_train_scaled, Y_train)

y_pred_log_best = log_best.predict(X_test_scaled)

print("Logistic (best params + balanced)")
print(classification_report(y_test, y_pred_log_best))

Logistic (best params + balanced)
              precision    recall  f1-score   support

           0       0.92      0.72      0.81      1036
           1       0.52      0.82      0.63       373

    accuracy                           0.75      1409
   macro avg       0.72      0.77      0.72      1409
weighted avg       0.81      0.75      0.76      1409



In [79]:
from sklearn.metrics import recall_score, f1_score, accuracy_score

def evaluate(name, y_true, y_pred):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "recall_1": recall_score(y_true, y_pred, pos_label=1),
        "f1_1": f1_score(y_true, y_pred, pos_label=1)
    }

results = []
results.append(evaluate("KNN base", y_test, y_pred_knn))
results.append(evaluate("Logistic balanced", y_test, y_pred_log))
results.append(evaluate("RF balanced", y_test, y_pred_rf_bal))
results.append(evaluate("Logistic + SMOTE", y_test, y_pred_log_smote))
results.append(evaluate("KNN best", y_test, y_pred_knn_best))
results.append(evaluate("Logistic best+balanced", y_test, y_pred_log_best))

pd.DataFrame(results).sort_values(by="recall_1", ascending=False)

,model,accuracy,recall_1,f1_1
3,Logistic + SMOTE,0.755855,0.833780,0.643892
1,Logistic balanced,0.748758,0.823056,0.634298
5,Logistic best+balanced,0.748758,0.823056,0.634298
4,KNN best,0.794180,0.544236,0.583333
0,KNN base,0.771469,0.520107,0.546479
2,RF balanced,0.791341,0.453083,0.534810
